# 01 — Database quality check

This notebook checks the quality of the NIR UCO database built in `00_build_database.ipynb`.

The goal is to verify:

- image-level metadata,
- object-level metadata,
- segmentation quality,
- object area distributions,
- spectral consistency,
- batch effects and potentially noisy images.

This notebook should not rebuild the database. It should only load the saved HDF5 database and produce QC tables / plots.

In [3]:
from __future__ import annotations

import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 250)

# ---------------------------------------------------------------------
# Project root detection
# ---------------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Launch the notebook from the project "
        "root or from the notebooks/ folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [4]:
from src.io.database_h5 import load_nir_uco_h5
from src.io.dataload import load_mat_file
from src.data.database import parse_image_key, preprocess_nir_uco_cube
from src.data.segmentation import segment_objects, make_reference_image

from src.visualization.plot_images import (
    plot_image2d,
    plot_label_overlay_from_image_db,
)

from src.visualization.plot_spectra import (
    plot_spectra,
    plot_spectral_distribution,
)

from src.visualization.plot_generic import (
    plot_bar_values,
    plot_counts_by_group,
)

from src.visualization.plot_objects import (
    plot_object_view,
    plot_object_grid,
    plot_object_area_distribution,
)

from src.utils import save_parquet, save_parquet_if_nonempty

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
# ---------------------------------------------------------------------
# Input database
# ---------------------------------------------------------------------
DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

RAW_MAT_PATH = (
    PROJECT_ROOT
    / "HSI Data"
    / "NIR camera UCO (889-1702 nm)"
    / "NIR_uco_sb.mat"
)
N_REMOVE_START = 6
N_STOP_END = None

SEGMENTATION_KWARGS = {
    "reference_method": "max",
    "threshold_method": "fixed",
    "tau_min": 0.02,
    "opening_radius": 0,
    "closing_radius": 1,
    "fill_holes": True,
    "min_distance": 10,
    "min_area": 10,
    "use_watershed": False,
}

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
RESULTS_DIR = PROJECT_ROOT / "results" / "01_quality_check"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_QC_PATH = RESULTS_DIR / "image_qc_summary.parquet"
OBJECT_QC_PATH = RESULTS_DIR / "object_qc_summary.parquet"
QC_FLAGS_PATH = RESULTS_DIR / "qc_flags.parquet"

# ---------------------------------------------------------------------
# QC parameters
# ---------------------------------------------------------------------
RUN_SEGMENTATION_QC_PLOTS = True
RUN_OBJECT_QC_PLOTS = True
RUN_SPECTRAL_QC_PLOTS = True
RUN_RAW_DB_COMPARISON = False

RECONSTRUCT_HEAVY_OBJECT_ARRAYS = True

N_EXAMPLE_IMAGES_PER_KIND = 2
N_OBJECTS_IN_GRID = 25
RANDOM_STATE = 42

# For spectral plots
MAX_OBJECT_SPECTRA_PER_GROUP = 80

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("IMAGE_QC_PATH:", IMAGE_QC_PATH)
print("OBJECT_QC_PATH:", OBJECT_QC_PATH)
print("QC_FLAGS_PATH:", QC_FLAGS_PATH)

DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\01_quality_check
IMAGE_QC_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\01_quality_check\image_qc_summary.parquet
OBJECT_QC_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\01_quality_check\object_qc_summary.parquet
QC_FLAGS_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\01_quality_check\qc_flags.parquet


In [7]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(
        f"Database not found: {DB_H5_PATH}\n"
        "Run 00_build_database.ipynb first."
    )

print("Database file found.")
print(f"File size: {DB_H5_PATH.stat().st_size / 1024**2:.2f} MB")

Database file found.
File size: 115.60 MB


In [8]:
object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=RECONSTRUCT_HEAVY_OBJECT_ARRAYS,
)

print(f"Number of images: {len(image_db)}")
print(f"Number of objects: {len(object_db)}")

print("\nFirst image keys:")
print(list(image_db.keys())[:10])

print("\nFirst object ids:")
print(list(object_db.keys())[:10])

Number of images: 48
Number of objects: 1262

First image keys:
['alm1pea1', 'alm1pea2', 'alm1pea3', 'alm1pea4', 'alm2pea1', 'alm2pea2', 'alm2pea3', 'alm2pea4', 'alm3pea1', 'alm3pea2']

First object ids:
['alm1pea1_obj001', 'alm1pea1_obj002', 'alm1pea1_obj003', 'alm1pea1_obj004', 'alm1pea1_obj005', 'alm1pea1_obj006', 'alm1pea1_obj007', 'alm1pea1_obj008', 'alm1pea1_obj009', 'alm1pea1_obj010']


In [9]:
image_rows = []

for image_key, img in image_db.items():
    cube = np.asarray(img["cube"])
    labels = np.asarray(img["labels"])

    unique_labels = np.unique(labels)
    object_labels = unique_labels[unique_labels > 0]

    wavelengths = img.get("wavelengths")
    has_wavelengths = wavelengths is not None and len(np.asarray(wavelengths)) > 0

    image_rows.append({
        "clean_key": image_key,
        "image_id": img.get("image_id"),
        "sample_kind": img.get("sample_kind"),
        "nut_type": img.get("nut_type"),
        "batch": img.get("batch"),
        "position_set": img.get("position_set"),
        "description": img.get("description"),

        "is_pure": bool(img.get("is_pure", False)),
        "is_mixture": bool(img.get("is_mixture", False)),
        "is_position_reference": bool(img.get("is_position_reference", False)),
        "is_unknown": bool(img.get("is_unknown", False)),

        "height": cube.shape[0],
        "width": cube.shape[1],
        "n_bands": cube.shape[2],
        "n_pixels_image": cube.shape[0] * cube.shape[1],

        "n_objects_recorded": int(img.get("n_objects", 0)),
        "n_labels_positive": int(len(object_labels)),
        "max_label": int(labels.max()) if labels.size else 0,

        "threshold": img.get("threshold"),
        "mask_area_pixels": int(np.asarray(img["mask"]).sum()) if "mask" in img else np.nan,
        "mask_area_ratio": (
            float(np.asarray(img["mask"]).sum() / labels.size)
            if "mask" in img and labels.size > 0
            else np.nan
        ),

        "has_wavelengths": bool(has_wavelengths),
        "data_mode": img.get("data_mode"),
    })

image_qc_df = (
    pd.DataFrame(image_rows)
    .sort_values(
        ["sample_kind", "nut_type", "batch", "position_set", "clean_key"],
        na_position="last",
    )
    .reset_index(drop=True)
)

display(image_qc_df)

,clean_key,image_id,sample_kind,nut_type,batch,position_set,description,is_pure,is_mixture,is_position_reference,is_unknown,height,width,n_bands,n_pixels_image,n_objects_recorded,n_labels_positive,max_label,threshold,mask_area_pixels,mask_area_ratio,has_wavelengths,data_mode
0,alm1pea1,alm1pea1_sb,mixture,mixture,NaN,NaN,mixture: almond batch 1 + peanut batch 1,False,True,False,False,370,318,63,117660,42,42,42,0.02,3722,0.031634,True,reflectance
1,alm1pea2,alm1pea2_sb,mixture,mixture,NaN,NaN,mixture: almond batch 1 + peanut batch 2,False,True,False,False,370,318,63,117660,40,40,40,0.02,2678,0.022760,True,reflectance
2,alm1pea3,alm1pea3_sb,mixture,mixture,NaN,NaN,mixture: almond batch 1 + peanut batch 3,False,True,False,False,370,318,63,117660,40,40,40,0.02,2350,0.019973,True,reflectance
3,alm1pea4,alm1pea4_sb,mixture,mixture,NaN,NaN,mixture: almond batch 1 + peanut batch 4,False,True,False,False,370,318,63,117660,27,27,27,0.02,2838,0.024120,True,reflectance
4,alm2pea1,alm2pea1_sb,mixture,mixture,NaN,NaN,mixture: almond batch 2 + peanut batch 1,False,True,False,False,370,318,63,117660,40,40,40,0.02,2417,0.020542,True,reflectance
5,alm2pea2,alm2pea2_sb,mixture,mixture,NaN,NaN,mixture: almond batch 2 + peanut batch 2,False,True,False,False,370,318,63,117660,40,40,40,0.02,2436,0.020704,True,reflectance
6,alm2pea3,alm2pea3_sb,mixture,mixture,NaN,NaN,mixture: almond batch 2 + peanut batch 3,False,True,False,False,370,318,63,117660,44,44,44,0.02,2391,0.020321,True,reflectance
7,alm2pea4,alm2pea4_sb,mixture,mixture,NaN,NaN,mixture: almond batch 2 + peanut batch 4,False,True,False,False,370,318,63,117660,33,33,33,0.02,3917,0.033291,True,reflectance
8,alm3pea1,alm3pea1_sb,mixture,mixture,NaN,NaN,mixture: almond batch 3 + peanut batch 1,False,True,False,False,370,318,63,117660,42,42,42,0.02,3190,0.027112,True,reflectance
9,alm3pea2,alm3pea2_sb,mixture,mixture,NaN,NaN,mixture: almond batch 3 + peanut batch 2,False,True,False,False,370,318,63,117660,39,39,39,0.02,2446,0.020789,True,reflectance


In [10]:
save_parquet(image_qc_df, IMAGE_QC_PATH)

print("Saved image QC summary:")
print(IMAGE_QC_PATH)

Saved image QC summary:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\01_quality_check\image_qc_summary.parquet


In [11]:
image_kind_summary = (
    image_qc_df
    .groupby(["sample_kind", "nut_type"], dropna=False)
    .agg(
        n_images=("clean_key", "count"),
        n_objects=("n_objects_recorded", "sum"),
        median_objects_per_image=("n_objects_recorded", "median"),
        min_objects_per_image=("n_objects_recorded", "min"),
        max_objects_per_image=("n_objects_recorded", "max"),
    )
    .reset_index()
    .sort_values(["sample_kind", "nut_type"])
)

image_kind_summary

,sample_kind,nut_type,n_images,n_objects,median_objects_per_image,min_objects_per_image,max_objects_per_image
0,mixture,mixture,20,722,37.5,25,44
1,position_reference,peanut,20,146,1.0,1,15
2,pure,almond,4,214,53.5,48,59
3,pure,peanut,4,180,49.0,29,53


In [12]:
image_batch_summary = (
    image_qc_df
    .groupby(["sample_kind", "nut_type", "batch"], dropna=False)
    .agg(
        n_images=("clean_key", "count"),
        n_objects=("n_objects_recorded", "sum"),
        median_objects_per_image=("n_objects_recorded", "median"),
        median_mask_area_ratio=("mask_area_ratio", "median"),
    )
    .reset_index()
    .sort_values(["sample_kind", "nut_type", "batch"], na_position="last")
)

image_batch_summary

,sample_kind,nut_type,batch,n_images,n_objects,median_objects_per_image,median_mask_area_ratio
0,mixture,mixture,NaN,20,722,37.5,0.026959
1,position_reference,peanut,1.0,5,47,15.0,0.006502
2,position_reference,peanut,2.0,5,47,15.0,0.006961
3,position_reference,peanut,3.0,5,47,15.0,0.007224
4,position_reference,peanut,4.0,5,5,1.0,0.001079
5,pure,almond,1.0,1,52,52.0,0.035492
6,pure,almond,2.0,1,59,59.0,0.031897
7,pure,almond,3.0,1,55,55.0,0.030724
8,pure,almond,4.0,1,48,48.0,0.044510
9,pure,peanut,1.0,1,46,46.0,0.027061


In [13]:
plot_counts_by_group(
    image_qc_df,
    group_col="sample_kind",
    category_col="nut_type",
    title="Image counts by sample kind and nut type",
    show=True,
)

plot_bar_values(
    x=image_qc_df["clean_key"],
    y=image_qc_df["n_objects_recorded"],
    title="Number of detected objects per image",
    x_title="image",
    y_title="n_objects",
    show=True,
)

In [14]:
image_warnings = []

for _, row in image_qc_df.iterrows():
    if row["n_objects_recorded"] == 0:
        image_warnings.append({
            "clean_key": row["clean_key"],
            "warning": "No object detected",
        })

    if row["n_objects_recorded"] != row["n_labels_positive"]:
        image_warnings.append({
            "clean_key": row["clean_key"],
            "warning": (
                f"n_objects_recorded={row['n_objects_recorded']} differs from "
                f"n_labels_positive={row['n_labels_positive']}"
            ),
        })

    if pd.isna(row["mask_area_ratio"]) or row["mask_area_ratio"] <= 0:
        image_warnings.append({
            "clean_key": row["clean_key"],
            "warning": "Empty or invalid mask area ratio",
        })

image_warnings_df = pd.DataFrame(image_warnings)

image_warnings_df

""


## 2. Object-level quality control

We now summarize all objects stored in `object_db`.

For each object, we check:

- source image,
- object label,
- batch,
- split,
- area,
- number of pixels,
- number of spectral bands,
- centroid,
- bounding box.

In [15]:
object_rows = []

for object_id, obj in object_db.items():
    centroid = obj.get("centroid", (np.nan, np.nan))
    bbox = obj.get("bbox", None)

    spectra = np.asarray(obj.get("spectra"))
    mean_spectrum = np.asarray(obj.get("mean_spectrum"))

    if bbox is not None:
        min_row, min_col, max_row, max_col = bbox
        bbox_height = int(max_row - min_row)
        bbox_width = int(max_col - min_col)
    else:
        bbox_height = np.nan
        bbox_width = np.nan

    object_rows.append({
        "object_id": object_id,
        "source_clean_key": obj.get("source_clean_key"),
        "source_image": obj.get("source_image"),
        "sample_kind": obj.get("sample_kind"),
        "image_nut_type": obj.get("image_nut_type"),
        "object_nut_type": obj.get("object_nut_type"),
        "batch": obj.get("batch"),
        "position_set": obj.get("position_set"),
        "split": obj.get("split"),

        "is_pure": bool(obj.get("is_pure", False)),
        "is_mixture": bool(obj.get("is_mixture", False)),
        "is_position_reference": bool(obj.get("is_position_reference", False)),
        "is_unknown": bool(obj.get("is_unknown", False)),

        "label_id": obj.get("label_id"),
        "object_index": obj.get("object_index"),

        "area_pixels": obj.get("area_pixels"),
        "n_pixels": obj.get("n_pixels"),
        "n_bands": obj.get("n_bands"),

        "spectra_shape": spectra.shape if spectra is not None else None,
        "mean_spectrum_length": len(mean_spectrum) if mean_spectrum is not None else np.nan,

        "centroid_row": centroid[0] if centroid is not None else np.nan,
        "centroid_col": centroid[1] if centroid is not None else np.nan,

        "bbox": bbox,
        "bbox_height": bbox_height,
        "bbox_width": bbox_width,
        "bbox_area": bbox_height * bbox_width if np.isfinite(bbox_height) and np.isfinite(bbox_width) else np.nan,

        "data_mode": obj.get("data_mode"),
    })

object_qc_df = (
    pd.DataFrame(object_rows)
    .sort_values(
        ["sample_kind", "object_nut_type", "batch", "source_clean_key", "object_index"],
        na_position="last",
    )
    .reset_index(drop=True)
)

display(object_qc_df)

,object_id,source_clean_key,source_image,sample_kind,image_nut_type,object_nut_type,batch,position_set,split,is_pure,is_mixture,is_position_reference,is_unknown,label_id,object_index,area_pixels,n_pixels,n_bands,spectra_shape,mean_spectrum_length,centroid_row,centroid_col,bbox,bbox_height,bbox_width,bbox_area,data_mode
0,alm1pea1_obj001,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,False,1,1,84,84,63,"(84, 63)",63,85.726190,45.107143,"(81, 39, 92, 51)",11,12,132,reflectance
1,alm1pea1_obj002,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,False,2,2,73,73,63,"(73, 63)",63,87.739726,123.767123,"(82, 120, 94, 128)",12,8,96,reflectance
2,alm1pea1_obj003,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,False,3,3,91,91,63,"(91, 63)",63,90.032967,157.340659,"(85, 152, 96, 164)",11,12,132,reflectance
3,alm1pea1_obj004,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,False,4,4,73,73,63,"(73, 63)",63,93.589041,90.164384,"(88, 87, 100, 96)",12,9,108,reflectance
4,alm1pea1_obj005,alm1pea1,alm1pea1_sb,mixture,mixture,unknown,NaN,NaN,projection,False,True,False,False,5,5,126,126,63,"(126, 63)",63,98.825397,70.579365,"(91, 65, 107, 76)",16,11,176,reflectance
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1257,peanut4_obj025,peanut4,peanut4_sb,pure,peanut,peanut,4.0,NaN,projection,True,False,False,False,25,25,195,195,63,"(195, 63)",63,286.112821,96.984615,"(278, 89, 295, 106)",17,17,289,reflectance
1258,peanut4_obj026,peanut4,peanut4_sb,pure,peanut,peanut,4.0,NaN,projection,True,False,False,False,26,26,99,99,63,"(99, 63)",63,294.868687,227.858586,"(287, 223, 302, 234)",15,11,165,reflectance
1259,peanut4_obj027,peanut4,peanut4_sb,pure,peanut,peanut,4.0,NaN,projection,True,False,False,False,27,27,124,124,63,"(124, 63)",63,295.508065,141.338710,"(288, 137, 304, 147)",16,10,160,reflectance
1260,peanut4_obj028,peanut4,peanut4_sb,pure,peanut,peanut,4.0,NaN,projection,True,False,False,False,28,28,88,88,63,"(88, 63)",63,303.704545,52.738636,"(299, 47, 311, 58)",12,11,132,reflectance


In [16]:
save_parquet(object_qc_df, OBJECT_QC_PATH)
print("Saved object QC summary:")
print(OBJECT_QC_PATH)

Saved object QC summary:
C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\01_quality_check\object_qc_summary.parquet


In [17]:
object_kind_summary = (
    object_qc_df
    .groupby(["sample_kind", "object_nut_type"], dropna=False)
    .agg(
        n_objects=("object_id", "count"),
        median_area=("area_pixels", "median"),
        min_area=("area_pixels", "min"),
        max_area=("area_pixels", "max"),
        median_n_pixels=("n_pixels", "median"),
    )
    .reset_index()
    .sort_values(["sample_kind", "object_nut_type"])
)

display(object_kind_summary)

,sample_kind,object_nut_type,n_objects,median_area,min_area,max_area,median_n_pixels
0,mixture,unknown,722,82.0,16,224,82.0
1,position_reference,peanut,146,64.5,12,221,64.5
2,pure,almond,214,76.0,21,188,76.0
3,pure,peanut,180,70.0,15,195,70.0


In [18]:
object_batch_summary = (
    object_qc_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .agg(
        n_objects=("object_id", "count"),
        median_area=("area_pixels", "median"),
        min_area=("area_pixels", "min"),
        max_area=("area_pixels", "max"),
        median_n_pixels=("n_pixels", "median"),
    )
    .reset_index()
    .sort_values(["sample_kind", "object_nut_type", "batch"], na_position="last")
)

object_batch_summary

,sample_kind,object_nut_type,batch,n_objects,median_area,min_area,max_area,median_n_pixels
0,mixture,unknown,NaN,722,82.0,16,224,82.0
1,position_reference,peanut,1.0,47,69.0,21,178,69.0
2,position_reference,peanut,2.0,47,65.0,12,167,65.0
3,position_reference,peanut,3.0,47,59.0,24,221,59.0
4,position_reference,peanut,4.0,5,127.0,123,158,127.0
5,pure,almond,1.0,52,78.0,23,176,78.0
6,pure,almond,2.0,59,61.0,33,117,61.0
7,pure,almond,3.0,55,63.0,21,138,63.0
8,pure,almond,4.0,48,111.0,64,188,111.0
9,pure,peanut,1.0,46,67.5,18,145,67.5


In [19]:
area_summary_df = (
    object_qc_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .agg(
        n_objects=("object_id", "count"),
        area_mean=("area_pixels", "mean"),
        area_std=("area_pixels", "std"),
        area_min=("area_pixels", "min"),
        area_q05=("area_pixels", lambda s: s.quantile(0.05)),
        area_median=("area_pixels", "median"),
        area_q95=("area_pixels", lambda s: s.quantile(0.95)),
        area_max=("area_pixels", "max"),
    )
    .reset_index()
    .sort_values(["sample_kind", "object_nut_type", "batch"], na_position="last")
)

area_summary_df

,sample_kind,object_nut_type,batch,n_objects,area_mean,area_std,area_min,area_q05,area_median,area_q95,area_max
0,mixture,unknown,NaN,722,88.326870,38.301289,16,34.05,82.0,161.95,224
1,position_reference,peanut,1.0,47,72.085106,31.005490,21,29.10,69.0,110.30,178
2,position_reference,peanut,2.0,47,66.914894,30.904361,12,22.20,65.0,117.70,167
3,position_reference,peanut,3.0,47,63.787234,31.092384,24,30.30,59.0,96.40,221
4,position_reference,peanut,4.0,5,136.600000,15.820872,123,123.60,127.0,156.20,158
5,pure,almond,1.0,52,80.307692,28.692787,23,44.20,78.0,135.30,176
6,pure,almond,2.0,59,63.610169,20.609687,33,35.00,61.0,92.20,117
7,pure,almond,3.0,55,65.727273,25.832405,21,28.10,63.0,106.60,138
8,pure,almond,4.0,48,109.104167,26.082716,64,70.45,111.0,144.65,188
9,pure,peanut,1.0,46,69.217391,27.080877,18,33.25,67.5,114.75,145


In [20]:
plot_counts_by_group(
    object_qc_df,
    group_col="sample_kind",
    category_col="object_nut_type",
    title="Object counts by sample kind and object label",
    show=True,
)

plot_object_area_distribution(
    object_qc_df,
    area_col="area_pixels",
    class_col="object_nut_type",
    batch_col="batch",
    kind="box",
    facet_by_batch=True,
    points="outliers",
    title="Object area distribution by class and batch",
    show=True,
)

In [21]:
object_warnings = []

for _, row in object_qc_df.iterrows():
    if row["area_pixels"] is None or pd.isna(row["area_pixels"]) or row["area_pixels"] <= 0:
        object_warnings.append({
            "object_id": row["object_id"],
            "warning": "Invalid object area",
        })

    if row["n_pixels"] != row["area_pixels"]:
        object_warnings.append({
            "object_id": row["object_id"],
            "warning": f"n_pixels={row['n_pixels']} differs from area_pixels={row['area_pixels']}",
        })

    if row["mean_spectrum_length"] != row["n_bands"]:
        object_warnings.append({
            "object_id": row["object_id"],
            "warning": (
                f"mean_spectrum_length={row['mean_spectrum_length']} differs "
                f"from n_bands={row['n_bands']}"
            ),
        })

    if row["bbox_area"] < row["area_pixels"]:
        object_warnings.append({
            "object_id": row["object_id"],
            "warning": "bbox_area smaller than object area",
        })

object_warnings_df = pd.DataFrame(object_warnings)

object_warnings_df

""


## 3. Segmentation visual quality control

We now inspect segmentation overlays for representative images.

The goal is to visually check whether the segmentation masks and labels correctly isolate the nuts.

In [22]:
rng = np.random.default_rng(RANDOM_STATE)

representative_image_keys = []

for sample_kind, sub in image_qc_df.groupby("sample_kind", dropna=False):
    candidate_keys = sub.loc[sub["n_objects_recorded"] > 0, "clean_key"].tolist()

    if len(candidate_keys) == 0:
        continue

    n_pick = min(N_EXAMPLE_IMAGES_PER_KIND, len(candidate_keys))
    picked = rng.choice(candidate_keys, size=n_pick, replace=False).tolist()
    representative_image_keys.extend(picked)

print("Representative images:")
print(representative_image_keys)

Representative images:
['alm1pea2', 'alm4pea4', 'pea2_pos4', 'pea4_pos5', 'peanut2', 'almond1']


In [23]:
if RUN_SEGMENTATION_QC_PLOTS:
    for image_key in representative_image_keys:
        print("Image:", image_key)

        plot_label_overlay_from_image_db(
            image_db=image_db,
            image_key=image_key,
            base="image_ref",
            title=f"Segmentation labels — {image_key}",
            show=True,
        )
else:
    print("Segmentation QC plots skipped.")

Image: alm1pea2


Image: alm4pea4


Image: pea2_pos4


Image: pea4_pos5


Image: peanut2


Image: almond1


### Compare to raw images

In [24]:
if RUN_RAW_DB_COMPARISON:
    raw_data = load_mat_file(RAW_MAT_PATH)

    RAW_KEY = "pea4_pos4_sb"
    DB_KEY = "pea4_pos4"

    if RAW_KEY not in raw_data:
        print(f"[WARNING] RAW_KEY not found in raw data: {RAW_KEY}")
    elif DB_KEY not in image_db:
        print(f"[WARNING] DB_KEY not found in image_db: {DB_KEY}")
    else:
        raw_cube = np.asarray(raw_data[RAW_KEY], dtype=float)
        raw_cube_processed = preprocess_nir_uco_cube(
            raw_cube,
            n_remove_start=N_REMOVE_START,
            n_stop_end=N_STOP_END,
        )
        raw_image_ref = np.nanmax(raw_cube_processed, axis=2)

        db_image_ref = np.asarray(image_db[DB_KEY]["image_ref"], dtype=float)

        plot_image2d(
            raw_image_ref,
            title=f"{RAW_KEY} from .mat — processed reference image",
            colorbar_title="max reflectance",
            show=True,
        )

        plot_image2d(
            db_image_ref,
            title=f"{DB_KEY} from DB — stored image_ref",
            colorbar_title="image_ref",
            show=True,
        )
else:
    print("Raw-vs-DB comparison skipped.")

Raw-vs-DB comparison skipped.


### Object level

In [25]:
if RUN_OBJECT_QC_PLOTS:
    for image_key in representative_image_keys:
        print("Objects from image:", image_key)

        try:
            plot_object_grid(
                object_db,
                source_image=image_key,
                title=f"Object grid — {image_key}",
                max_objects=N_OBJECTS_IN_GRID,
                n_cols=5,
                show=True,
            )
        except Exception as exc:
            print(f"[WARNING] Could not plot object grid for {image_key}: {exc!r}")
else:
    print("Object QC plots skipped.")

Objects from image: alm1pea2


Objects from image: alm4pea4


Objects from image: pea2_pos4


Objects from image: pea4_pos5


Objects from image: peanut2


Objects from image: almond1


In [26]:
# Select a few representative objects:
# - largest objects
# - smallest objects
# - one per main class if available

largest_objects = (
    object_qc_df
    .sort_values("area_pixels", ascending=False)
    .head(3)["object_id"]
    .tolist()
)

smallest_objects = (
    object_qc_df
    .sort_values("area_pixels", ascending=True)
    .head(3)["object_id"]
    .tolist()
)

class_examples = []
for label, sub in object_qc_df.groupby("object_nut_type", dropna=False):
    if len(sub) > 0:
        class_examples.append(sub.iloc[0]["object_id"])

objects_to_view = list(dict.fromkeys(largest_objects + smallest_objects + class_examples))

print("Objects selected for detailed view:")
print(objects_to_view)

Objects selected for detailed view:
['alm5pea3_obj028', 'pea3_pos5_obj001', 'alm5pea3_obj036', 'pea2_pos2_obj015', 'peanut2_obj004', 'alm2pea2_obj040', 'almond1_obj001', 'pea1_pos1_obj001', 'alm1pea1_obj001']


In [27]:
if RUN_SPECTRAL_QC_PLOTS:
    for object_id in objects_to_view:
        print("Object:", object_id)

        try:
            plot_object_view(
                object_db,
                object_id=object_id,
                spectrum_field="mean_spectrum",
                show_spectrum=True,
                show_std=True,
                show=True,
            )
        except Exception as exc:
            print(f"[WARNING] Could not plot object {object_id}: {exc!r}")
else:
    print("Spectral QC plots skipped.")

Object: alm5pea3_obj028


Object: pea3_pos5_obj001


Object: alm5pea3_obj036


Object: pea2_pos2_obj015


Object: peanut2_obj004


Object: alm2pea2_obj040


Object: almond1_obj001


Object: pea1_pos1_obj001


Object: alm1pea1_obj001


## 4. Spectral quality control

We inspect object spectra by class, batch and image type.

Important checks:

- Do almond and peanut spectra differ?
- Are some batches shifted or noisier?
- Is batch 3 particularly noisy?
- Are mixture and position-reference spectra coherent?

In [28]:
spectral_rows = []
spectra_list = []

for object_id, obj in object_db.items():
    spectrum = np.asarray(obj["mean_spectrum"], dtype=float)
    spectra_list.append(spectrum)

    spectral_rows.append({
        "object_id": object_id,
        "source_clean_key": obj.get("source_clean_key"),
        "sample_kind": obj.get("sample_kind"),
        "object_nut_type": obj.get("object_nut_type"),
        "batch": obj.get("batch"),
        "split": obj.get("split"),
        "area_pixels": obj.get("area_pixels"),
        "n_pixels": obj.get("n_pixels"),
        "spectrum_mean": float(np.nanmean(spectrum)),
        "spectrum_std": float(np.nanstd(spectrum)),
        "spectrum_min": float(np.nanmin(spectrum)),
        "spectrum_max": float(np.nanmax(spectrum)),
        "spectrum_range": float(np.nanmax(spectrum) - np.nanmin(spectrum)),
        "spectrum_nan_rate": float(np.mean(~np.isfinite(spectrum))),
    })

X_mean = np.vstack(spectra_list)
spectral_qc_df = pd.DataFrame(spectral_rows)

display(spectral_qc_df)
print("X_mean shape:", X_mean.shape)

,object_id,source_clean_key,sample_kind,object_nut_type,batch,split,area_pixels,n_pixels,spectrum_mean,spectrum_std,spectrum_min,spectrum_max,spectrum_range,spectrum_nan_rate
0,alm1pea1_obj001,alm1pea1,mixture,unknown,NaN,projection,84,84,0.294065,0.081082,0.140729,0.396776,0.256047,0.0
1,alm1pea1_obj002,alm1pea1,mixture,unknown,NaN,projection,73,73,0.324405,0.088211,0.141958,0.438084,0.296125,0.0
2,alm1pea1_obj003,alm1pea1,mixture,unknown,NaN,projection,91,91,0.387276,0.104366,0.169208,0.518123,0.348915,0.0
3,alm1pea1_obj004,alm1pea1,mixture,unknown,NaN,projection,73,73,0.379292,0.084707,0.183080,0.485413,0.302333,0.0
4,alm1pea1_obj005,alm1pea1,mixture,unknown,NaN,projection,126,126,0.431196,0.107845,0.208009,0.566116,0.358108,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1257,peanut4_obj025,peanut4,pure,peanut,4.0,projection,195,195,0.368879,0.099566,0.150785,0.497247,0.346462,0.0
1258,peanut4_obj026,peanut4,pure,peanut,4.0,projection,99,99,0.342355,0.091527,0.136313,0.458314,0.322001,0.0
1259,peanut4_obj027,peanut4,pure,peanut,4.0,projection,124,124,0.374064,0.097510,0.164787,0.499281,0.334494,0.0
1260,peanut4_obj028,peanut4,pure,peanut,4.0,projection,88,88,0.367983,0.079895,0.163482,0.465191,0.301709,0.0


X_mean shape: (1262, 63)


In [29]:
spectral_summary_df = (
    spectral_qc_df
    .groupby(["sample_kind", "object_nut_type", "batch"], dropna=False)
    .agg(
        n_objects=("object_id", "count"),
        spectrum_mean_mean=("spectrum_mean", "mean"),
        spectrum_mean_std=("spectrum_mean", "std"),
        spectrum_std_mean=("spectrum_std", "mean"),
        spectrum_range_median=("spectrum_range", "median"),
        spectrum_nan_rate_max=("spectrum_nan_rate", "max"),
    )
    .reset_index()
    .sort_values(["sample_kind", "object_nut_type", "batch"], na_position="last")
)

spectral_summary_df

,sample_kind,object_nut_type,batch,n_objects,spectrum_mean_mean,spectrum_mean_std,spectrum_std_mean,spectrum_range_median,spectrum_nan_rate_max
0,mixture,unknown,NaN,722,0.341730,0.057270,0.090197,0.299310,0.0
1,position_reference,peanut,1.0,47,0.354138,0.045341,0.085986,0.314063,0.0
2,position_reference,peanut,2.0,47,0.349743,0.045185,0.081676,0.288205,0.0
3,position_reference,peanut,3.0,47,0.322530,0.048476,0.081949,0.285239,0.0
4,position_reference,peanut,4.0,5,0.390380,0.032674,0.101015,0.329067,0.0
5,pure,almond,1.0,52,0.343849,0.055905,0.089280,0.296547,0.0
6,pure,almond,2.0,59,0.317700,0.051632,0.082859,0.273392,0.0
7,pure,almond,3.0,55,0.334150,0.059558,0.085308,0.281535,0.0
8,pure,almond,4.0,48,0.361010,0.058284,0.099632,0.322265,0.0
9,pure,peanut,1.0,46,0.356811,0.054712,0.084000,0.292338,0.0


In [30]:
# Use wavelength axis from the first object if available.
first_object = next(iter(object_db.values()))
wavelengths = first_object.get("wavelengths")

if wavelengths is not None:
    wavelengths = np.asarray(wavelengths)
    if wavelengths.size == 0:
        wavelengths = None

if wavelengths is None:
    print("No wavelength axis found. Plots will use band indices.")
else:
    print("Wavelength axis found.")
    print("n_wavelengths:", len(wavelengths))
    print("first:", wavelengths[:5])
    print("last:", wavelengths[-5:])

Wavelength axis found.
n_wavelengths: 63
first: [ 960.73529412  972.69117647  984.64705882  996.60294118 1008.55882353]
last: [1654.17647059 1666.13235294 1678.08823529 1690.04411765 1702.        ]


In [31]:
pure_objects_df = spectral_qc_df[
    spectral_qc_df["sample_kind"].eq("pure")
].copy()

pure_object_ids = pure_objects_df["object_id"].tolist()

# Optional sampling for readability
if len(pure_object_ids) > 2 * MAX_OBJECT_SPECTRA_PER_GROUP:
    sampled_ids = []
    for label, sub in pure_objects_df.groupby("object_nut_type"):
        sampled = sub.sample(
            n=min(MAX_OBJECT_SPECTRA_PER_GROUP, len(sub)),
            random_state=RANDOM_STATE,
        )
        sampled_ids.extend(sampled["object_id"].tolist())
    pure_object_ids = sampled_ids

plot_spectra(
    object_db,
    keys=pure_object_ids,
    spectrum_field="mean_spectrum",
    reducer="mean_std",
    title="Pure object spectra — mean ± std by object label",
    y_title="Reflectance",
    show=True,
)

In [32]:
almond_ids = spectral_qc_df[
    spectral_qc_df["sample_kind"].eq("pure")
    & spectral_qc_df["object_nut_type"].eq("almond")
]["object_id"].tolist()

if len(almond_ids) > 0:
    X_almond = np.vstack([object_db[oid]["mean_spectrum"] for oid in almond_ids])
    labels_almond = np.asarray([
        f"batch {object_db[oid].get('batch')}"
        for oid in almond_ids
    ])

    plot_spectra(
        X_almond,
        wavelengths=wavelengths,
        labels=labels_almond,
        reducer="mean_std",
        title="Pure almond spectra by batch — mean ± std",
        y_title="Reflectance",
        show=True,
    )
else:
    print("No pure almond object found.")

In [33]:
peanut_ids = spectral_qc_df[
    spectral_qc_df["sample_kind"].eq("pure")
    & spectral_qc_df["object_nut_type"].eq("peanut")
]["object_id"].tolist()

if len(peanut_ids) > 0:
    X_peanut = np.vstack([object_db[oid]["mean_spectrum"] for oid in peanut_ids])
    labels_peanut = np.asarray([
        f"batch {object_db[oid].get('batch')}"
        for oid in peanut_ids
    ])

    plot_spectra(
        X_peanut,
        wavelengths=wavelengths,
        labels=labels_peanut,
        reducer="mean_std",
        title="Pure peanut spectra by batch — mean ± std",
        y_title="Reflectance",
        show=True,
    )
else:
    print("No pure peanut object found.")

In [34]:
batch3_ids = spectral_qc_df[
    spectral_qc_df["sample_kind"].eq("pure")
    & spectral_qc_df["batch"].eq(3)
]["object_id"].tolist()

if len(batch3_ids) > 0:
    X_batch3 = np.vstack([object_db[oid]["mean_spectrum"] for oid in batch3_ids])
    labels_batch3 = np.asarray([
        object_db[oid].get("object_nut_type", "unknown")
        for oid in batch3_ids
    ])

    plot_spectra(
        X_batch3,
        wavelengths=wavelengths,
        labels=labels_batch3,
        reducer="mean_std",
        title="Pure batch 3 spectra — mean ± std by class",
        y_title="Reflectance",
        show=True,
    )
else:
    print("No pure batch 3 object found.")

In [35]:
# Simple spectral outlier score based on object mean spectrum global mean and std.
# This is not a statistical outlier model; it is only a QC flag.

spectral_qc_df["z_spectrum_mean"] = (
    spectral_qc_df["spectrum_mean"] - spectral_qc_df["spectrum_mean"].mean()
) / (spectral_qc_df["spectrum_mean"].std() + 1e-12)

spectral_qc_df["z_spectrum_std"] = (
    spectral_qc_df["spectrum_std"] - spectral_qc_df["spectrum_std"].mean()
) / (spectral_qc_df["spectrum_std"].std() + 1e-12)

spectral_qc_df["spectral_outlier_score"] = (
    spectral_qc_df["z_spectrum_mean"].abs()
    + spectral_qc_df["z_spectrum_std"].abs()
)

spectral_outlier_candidates = (
    spectral_qc_df
    .sort_values("spectral_outlier_score", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

spectral_outlier_candidates

,object_id,source_clean_key,sample_kind,object_nut_type,batch,split,area_pixels,n_pixels,spectrum_mean,spectrum_std,spectrum_min,spectrum_max,spectrum_range,spectrum_nan_rate,z_spectrum_mean,z_spectrum_std,spectral_outlier_score
0,alm3pea2_obj026,alm3pea2,mixture,unknown,NaN,projection,94,94,0.642612,0.105573,0.371854,0.768012,0.396158,0.0,5.390015,1.223493,6.613508
1,alm3pea4_obj008,alm3pea4,mixture,unknown,NaN,projection,150,150,0.620685,0.110105,0.359985,0.756405,0.396420,0.0,4.995496,1.548924,6.544420
2,almond4_obj004,almond4,pure,almond,4.0,projection,139,139,0.535817,0.128987,0.273666,0.695431,0.421765,0.0,3.468501,2.905028,6.373529
3,alm2pea4_obj016,alm2pea4,mixture,unknown,NaN,projection,123,123,0.523585,0.124588,0.261741,0.678022,0.416281,0.0,3.248423,2.589102,5.837525
4,alm1pea4_obj022,alm1pea4,mixture,unknown,NaN,projection,125,125,0.515002,0.124196,0.261874,0.674284,0.412411,0.0,3.093988,2.560981,5.654969
5,alm1pea4_obj013,alm1pea4,mixture,unknown,NaN,projection,118,118,0.520416,0.118792,0.271124,0.670326,0.399202,0.0,3.191400,2.172829,5.364228
6,alm3pea3_obj037,alm3pea3,mixture,unknown,NaN,projection,111,111,0.489224,0.124570,0.250539,0.638723,0.388184,0.0,2.630164,2.587823,5.217987
7,alm1pea4_obj006,alm1pea4,mixture,unknown,NaN,projection,100,100,0.508125,0.115134,0.236529,0.648657,0.412128,0.0,2.970257,1.910105,4.880362
8,alm3pea1_obj020,alm3pea1,mixture,unknown,NaN,projection,96,96,0.543604,0.105859,0.281070,0.669491,0.388421,0.0,3.608600,1.244024,4.852624
9,alm4pea1_obj002,alm4pea1,mixture,unknown,NaN,projection,140,140,0.472479,0.123387,0.222683,0.626021,0.403338,0.0,2.328893,2.502868,4.831760


In [36]:
outlier_ids = spectral_outlier_candidates["object_id"].tolist()

if len(outlier_ids) > 0:
    plot_spectra(
        object_db,
        keys=outlier_ids,
        spectrum_field="mean_spectrum",
        reducer="none",
        title="Top spectral outlier candidates",
        y_title="Reflectance",
        show=True,
    )

## 5. Image-level spectral distribution

We inspect selected full-image spectral distributions by sampling pixels from the hyperspectral cubes.

In [37]:
for image_key in representative_image_keys:
    print("Spectral distribution:", image_key)

    cube = image_db[image_key]["cube"]
    wavelengths_img = image_db[image_key].get("wavelengths", wavelengths)

    try:
        object_mask = (
            np.asarray(image_db[image_key]["labels"]) > 0
        )
        plot_spectral_distribution(
            cube,
            mask=object_mask,
            title=f"Pixel spectral distribution — {image_key}",
            n_pixels=2000,
            wavelengths=wavelengths_img,
            random_state=RANDOM_STATE,
            y_title="Reflectance",
            reducer="mean_std",
            show=True,
        )
    except Exception as exc:
        print(f"[WARNING] Could not plot spectral distribution for {image_key}: {exc!r}")

Spectral distribution: alm1pea2


Spectral distribution: alm4pea4


Spectral distribution: pea2_pos4


Spectral distribution: pea4_pos5


Spectral distribution: peanut2


Spectral distribution: almond1


## 6. Consistency checks

We verify that key fields required by downstream notebooks are present.

The next notebooks need:

- `cube`, `image_ref`, `mask`, `labels` in `image_db`,
- `spectra`, `mean_spectrum`, `median_spectrum`, `std_spectrum`,
- object geometry: `mask`, `mask_global`, `positions_global`, `bbox`, `centroid`,
- metadata: `object_nut_type`, `sample_kind`, `batch`, `split`.

In [38]:
required_image_fields = [
    "cube",
    "image_ref",
    "mask",
    "labels",
    "clean_key",
    "sample_kind",
    "nut_type",
    "n_objects",
    "object_ids",
]

required_object_fields = [
    "object_id",
    "source_clean_key",
    "sample_kind",
    "object_nut_type",
    "batch",
    "split",
    "bbox",
    "centroid",
    "area_pixels",
    "mask",
    "mask_global",
    "positions_global",
    "spectra",
    "mean_spectrum",
    "median_spectrum",
    "std_spectrum",
]

missing_rows = []

for image_key, img in image_db.items():
    missing = [field for field in required_image_fields if field not in img]
    if missing:
        missing_rows.append({
            "record_type": "image",
            "record_id": image_key,
            "missing_fields": missing,
        })

for object_id, obj in object_db.items():
    missing = [field for field in required_object_fields if field not in obj]
    if missing:
        missing_rows.append({
            "record_type": "object",
            "record_id": object_id,
            "missing_fields": missing,
        })

missing_fields_df = pd.DataFrame(missing_rows)

display(missing_fields_df)

if missing_fields_df.empty:
    print("All required fields are present.")
else:
    print("[WARNING] Some required fields are missing.")

""


All required fields are present.


In [39]:
shape_rows = []

for object_id, obj in object_db.items():
    source_key = obj.get("source_clean_key")
    img = image_db.get(source_key)

    spectra = np.asarray(obj["spectra"])
    mean_spectrum = np.asarray(obj["mean_spectrum"])
    median_spectrum = np.asarray(obj["median_spectrum"])
    std_spectrum = np.asarray(obj["std_spectrum"])
    positions_global = np.asarray(obj["positions_global"])

    row = {
        "object_id": object_id,
        "source_clean_key": source_key,
        "spectra_shape": spectra.shape,
        "positions_shape": positions_global.shape,
        "mean_spectrum_length": len(mean_spectrum),
        "median_spectrum_length": len(median_spectrum),
        "std_spectrum_length": len(std_spectrum),
        "n_pixels_recorded": obj.get("n_pixels"),
        "area_pixels": obj.get("area_pixels"),
        "n_bands_recorded": obj.get("n_bands"),
        "ok_spectra_pixels": spectra.shape[0] == obj.get("n_pixels"),
        "ok_positions_pixels": positions_global.shape[0] == obj.get("n_pixels"),
        "ok_mean_length": len(mean_spectrum) == obj.get("n_bands"),
        "ok_median_length": len(median_spectrum) == obj.get("n_bands"),
        "ok_std_length": len(std_spectrum) == obj.get("n_bands"),
    }

    if img is not None:
        cube = np.asarray(img["cube"])
        row["image_n_bands"] = cube.shape[2]
        row["ok_object_image_bands"] = obj.get("n_bands") == cube.shape[2]
    else:
        row["image_n_bands"] = np.nan
        row["ok_object_image_bands"] = False

    shape_rows.append(row)

shape_check_df = pd.DataFrame(shape_rows)

bad_shape_df = shape_check_df[
    ~shape_check_df[
        [
            "ok_spectra_pixels",
            "ok_positions_pixels",
            "ok_mean_length",
            "ok_median_length",
            "ok_std_length",
            "ok_object_image_bands",
        ]
    ].all(axis=1)
].copy()

display(bad_shape_df)

if bad_shape_df.empty:
    print("All object shape checks passed.")
else:
    print("[WARNING] Some object shape checks failed.")

,object_id,source_clean_key,spectra_shape,positions_shape,mean_spectrum_length,median_spectrum_length,std_spectrum_length,n_pixels_recorded,area_pixels,n_bands_recorded,ok_spectra_pixels,ok_positions_pixels,ok_mean_length,ok_median_length,ok_std_length,image_n_bands,ok_object_image_bands


All object shape checks passed.


In [40]:
qc_flag_parts = []

if len(image_warnings_df) > 0:
    tmp = image_warnings_df.copy()
    tmp["record_type"] = "image"
    tmp = tmp.rename(columns={"clean_key": "record_id"})
    tmp["flag_type"] = "image_warning"
    qc_flag_parts.append(tmp[["record_type", "record_id", "flag_type", "warning"]])

if len(object_warnings_df) > 0:
    tmp = object_warnings_df.copy()
    tmp["record_type"] = "object"
    tmp = tmp.rename(columns={"object_id": "record_id"})
    tmp["flag_type"] = "object_warning"
    qc_flag_parts.append(tmp[["record_type", "record_id", "flag_type", "warning"]])

if len(missing_fields_df) > 0:
    tmp = missing_fields_df.copy()
    tmp["flag_type"] = "missing_fields"
    tmp["warning"] = tmp["missing_fields"].astype(str)
    qc_flag_parts.append(tmp[["record_type", "record_id", "flag_type", "warning"]])

if len(bad_shape_df) > 0:
    tmp = bad_shape_df.copy()
    tmp["record_type"] = "object"
    tmp = tmp.rename(columns={"object_id": "record_id"})
    tmp["flag_type"] = "bad_shape"
    tmp["warning"] = "Object shape consistency check failed"
    qc_flag_parts.append(tmp[["record_type", "record_id", "flag_type", "warning"]])

qc_flags_df = (
    pd.concat(qc_flag_parts, ignore_index=True, sort=False)
    if qc_flag_parts
    else pd.DataFrame(columns=["record_type", "record_id", "flag_type", "warning"])
)

display(qc_flags_df)

saved_qc_flags_path = save_parquet_if_nonempty(qc_flags_df, QC_FLAGS_PATH)

if saved_qc_flags_path is None:
    print("No QC flag saved because no warning was detected.")
else:
    print("Saved QC flags:")
    print(saved_qc_flags_path)

,record_type,record_id,flag_type,warning


No QC flag saved because no warning was detected.


In [41]:
modelling_summary = (
    object_qc_df
    .groupby(["split", "sample_kind", "object_nut_type", "batch"], dropna=False)
    .agg(
        n_objects=("object_id", "count"),
        median_area=("area_pixels", "median"),
    )
    .reset_index()
    .sort_values(["split", "sample_kind", "object_nut_type", "batch"], na_position="last")
)

modelling_summary

,split,sample_kind,object_nut_type,batch,n_objects,median_area
0,projection,mixture,unknown,NaN,722,82.0
1,projection,position_reference,peanut,1.0,47,69.0
2,projection,position_reference,peanut,2.0,47,65.0
3,projection,position_reference,peanut,3.0,47,59.0
4,projection,position_reference,peanut,4.0,5,127.0
5,projection,pure,almond,1.0,52,78.0
6,projection,pure,almond,2.0,59,61.0
7,projection,pure,almond,3.0,55,63.0
8,projection,pure,almond,4.0,48,111.0
9,projection,pure,peanut,1.0,46,67.5


In [42]:
availability = {
    "pure_peanut_batches": sorted(
        object_qc_df.loc[
            object_qc_df["sample_kind"].eq("pure")
            & object_qc_df["object_nut_type"].eq("peanut")
            & object_qc_df["batch"].notna(),
            "batch",
        ].unique().tolist()
    ),
    "pure_almond_batches": sorted(
        object_qc_df.loc[
            object_qc_df["sample_kind"].eq("pure")
            & object_qc_df["object_nut_type"].eq("almond")
            & object_qc_df["batch"].notna(),
            "batch",
        ].unique().tolist()
    ),
    "n_mixture_objects": int(
        object_qc_df["sample_kind"].eq("mixture").sum()
    ),
    "n_position_reference_objects": int(
        object_qc_df["sample_kind"].eq("position_reference").sum()
    ),
}

availability

{'pure_peanut_batches': [1.0, 2.0, 3.0, 4.0],
 'pure_almond_batches': [1.0, 2.0, 3.0, 4.0],
 'n_mixture_objects': 722,
 'n_position_reference_objects': 146}

In [43]:
qc_report = {
    "db_h5_path": str(DB_H5_PATH),
    "n_images": int(len(image_db)),
    "n_objects": int(len(object_db)),

    "n_image_warnings": int(len(image_warnings_df)),
    "n_object_warnings": int(len(object_warnings_df)),
    "n_missing_field_records": int(len(missing_fields_df)),
    "n_bad_shape_records": int(len(bad_shape_df)),
    "n_qc_flags": int(len(qc_flags_df)),

    "image_qc": str(IMAGE_QC_PATH),
    "object_qc": str(OBJECT_QC_PATH),
    "qc_flags": str(QC_FLAGS_PATH) if len(qc_flags_df) > 0 else None,

    "availability": availability,
}

display(pd.DataFrame([qc_report]))

,db_h5_path,n_images,n_objects,n_image_warnings,n_object_warnings,n_missing_field_records,n_bad_shape_records,n_qc_flags,image_qc,object_qc,qc_flags,availability
0,C:\Users\alixg\OneDrive - Université Paris-Dau...,48,1262,0,0,0,0,0,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,None,"{'pure_peanut_batches': [1.0, 2.0, 3.0, 4.0], ..."


In [44]:
print("01_database_quality_check.ipynb completed.")
print()
print("Essential outputs:")
print(" -", IMAGE_QC_PATH)
print(" -", OBJECT_QC_PATH)

if len(qc_flags_df) > 0:
    print(" -", QC_FLAGS_PATH)

print()
print("Warnings:")
print(f" - image warnings: {len(image_warnings_df)}")
print(f" - object warnings: {len(object_warnings_df)}")
print(f" - missing field records: {len(missing_fields_df)}")
print(f" - bad shape records: {len(bad_shape_df)}")
print(f" - total QC flags: {len(qc_flags_df)}")
print()
print("Next notebook:")
print("02_matrices_preprocessing.ipynb")

01_database_quality_check.ipynb completed.

Essential outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\01_quality_check\image_qc_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\01_quality_check\object_qc_summary.parquet

Warnings:
 - image warnings: 0
 - object warnings: 0
 - missing field records: 0
 - bad shape records: 0
 - total QC flags: 0

Next notebook:
02_matrices_preprocessing.ipynb
